**REGEX**

In [ ]:
import re

In [ ]:
def extract_dimensions(product_name):
    pattern = r"(\d+(\.\d+)?\s*(mm|cm|inch|mm|L|kg|m|ft|\"|in|lb))"
    dimensions = re.findall(pattern, product_name.lower())
    return [dim[0] for dim in dimensions]

In [ ]:
product_names = [
    "THERMOCOL IRIS 15 L",
    "AL SHEET 1.05 MM CF",
    "THERMOCOL COOLGALE PRIME 48",
    "GROMMET EPDM ID25.4MM I/O PIPE",
    "GROMMET EPDM OD20MM HC POWER CORD",
    "NUT BOLT M6X20 MM-SINGLE WASHER"
]
for product in product_names:
    print(f"Product: {product}, Extracted Dimensions: {extract_dimensions(product)}")

**NLTK**

In [ ]:
import nltk
from nltk import word_tokenize, pos_tag, ne_chunk
from nltk.tree import Tree

In [ ]:
nltk.download('punkt')
nltk.download('averaged_perceptron_tagger')
nltk.download('maxent_ne_chunker')
nltk.download('words')

In [ ]:
def extract_named_entities(text):
    tokens = word_tokenize(text)
    pos_tags = pos_tag(tokens) 
    tree = ne_chunk(pos_tags)   

    named_entities = []
    for subtree in tree:
        if isinstance(subtree, Tree):
            entity_name = " ".join(token for token, pos in subtree.leaves())
            entity_type = subtree.label() 
            named_entities.append((entity_name, entity_type))
    return named_entities

In [ ]:
product_names = [
    "THERMOCOL IRIS 15 L",
    "AL SHEET 1.05 MM CF",
    "THERMOCOL COOLGALE PRIME 48",
    "GROMMET EPDM ID25.4MM I/O PIPE",
    "GROMMET EPDM OD20MM HC POWER CORD"
]


In [ ]:
for product in product_names:
    entities = extract_named_entities(product)
    print(f"Entities in '{product}': {entities}")

**SPACY**

In [ ]:
import json
import random
import spacy
from spacy.training import Example
from spacy.util import minibatch
import pandas as pd

In [ ]:
with open('annotations.json') as fp:
    training_data = json.load(fp)

In [ ]:
nlp = spacy.blank('en')

In [ ]:
ner = nlp.add_pipe('ner', last=True)

In [ ]:
for label in training_data['classes']:
    ner.add_label(label)

In [ ]:
num_iterations = 5
num_epochs = 10
batch_size = 8

In [ ]:
for i in range(num_iterations):
    optimizer = nlp.begin_training()

    examples = []
    for text, annotations in training_data['annotations']:
        example = Example.from_dict(nlp.make_doc(text), annotations)
        examples.append(example)
    nlp.initialize(lambda: examples)

    for epoch in range(num_epochs):
        random.shuffle(examples)
        batches = minibatch(examples, size=batch_size)
        for batch in batches:
            nlp.update(batch, sgd=optimizer)

In [ ]:
from transformers import AutoModel
model = AutoModel.from_pretrained("allenai/scibert_scivocab_uncased")

In [ ]:
from transformers import AutoTokenizer, AutoModelForTokenClassification, TrainingArguments, Trainer
from transformers import DataCollatorForTokenClassification

In [ ]:
from transformers import pipeline
import pandas as pd

In [ ]:
ner_pipeline = pipeline("ner", model="allenai/scibert_scivocab_uncased", tokenizer="allenai/scibert_scivocab_uncased")

In [ ]:
product_names = [
    "THERMOCOL IRIS 15 L",
    "AL SHEET 1.05 MM CF",
    "THERMOCOL COOLGALE PRIME 48",
    "GROMMET EPDM ID25.4MM I/O PIPE",
    "GROMMET EPDM OD20MM HC POWER CORD",
    "EPDM CLOSED GASKET",
    "GROMMET EPDM ABSORBER-BIG",
    "GROMMET EPDM ABSORBER-SMALL",
    "EPDM THREAD CAP EPDM-M6",
    "SILICON GASKET-VRV",
    "SS REDUCER TEE 3/4\"-1/2\"",
    "SS TEE 3/4\"",
    "RIVET 1/4 X 1/8 INCH",
    "GI SCREW NO.6X13MM",
    "STAND MS 300 LPD WIN HOT E",
    "TUBE SUPPORT 300 LPD WIN HOT E",
    "STAND MS 100 LPD W H PLUS",
    "TUBE SUPPORT MS 100 LPD W H PLUS",
    "TUBE SUPPORT MS PC V HOT 100 PLUS PR",
    "MS STAND 100 LPD V HOT PLUS PR",
    "STAND MS WH ECO 150 LPD",
    "AL END COVER 2 500 WH-H",
    'CLAMP SS FOR TANK GLADO 10/15 L'
]

In [ ]:
def ner_on_products(product_names):
    results = []
    for product in product_names:
        entities = ner_pipeline(product)
        extracted_entities = {
            "Product": product,
            "Entities": entities
        }
        results.append(extracted_entities)
    return results

In [ ]:
ner_results = ner_on_products(product_names)

In [ ]:
df = pd.DataFrame(ner_results)

In [ ]:
df['Entities'][0]

In [ ]:
df['Entities'][22]